In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ruta_actual = Path(__file__).resolve().parent

# 2. Sube dos carpetas hacia arriba 
ruta_archivo = ruta_actual.parent.parent / "17h55m39s-12-May.txt"
# Leer el archivo sin encabezados
df = pd.read_csv(
    ruta_archivo,
    sep=r'\s+',  # Separador
    header=None,  # Sin encabezados
    names=['tiempo', 'col2', 'actividad_extracelular', 'intra_LP', 'intra_PD', 'col6'],
    usecols=['tiempo', 'actividad_extracelular', 'intra_LP', 'intra_PD'] 
)

print(df.head())
print(f"\nDimensiones: {df.shape}")

In [ ]:
# Quiero ahora representarlo gráficamente para ver si los datos parecen correctos

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(df['tiempo']-df['tiempo'].min(), df['actividad_extracelular'], label='Actividad Extracelular')
plt.plot(df['tiempo']-df['tiempo'].min(), df['intra_LP'], label='Neurona LP')
plt.plot(df['tiempo']-df['tiempo'].min(), df['intra_PD'], label='Neurona PD')
plt.xlabel('Tiempo (ms)')
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt
from scipy import signal as sp_signal
def detrend_mantener_nivel(señal):
    """
    Elimina tendencia pero mantiene el nivel medio original
    """
    media_original = np.mean(señal)
    señal_detrend = sp_signal.detrend(señal, type='linear')
    señal_ajustada = señal_detrend + media_original
    return señal_ajustada

def filtro_media_movil_pandas(data, window_size):
    """
    Aplica media móvil manteniendo la longitud original.
    Los bordes usan menos puntos pero no hay artefactos severos.
    """
    import pandas as pd
    
    if not isinstance(data, pd.Series):
        data = pd.Series(data)
    
    # center=True centra la ventana
    # min_periods=1 permite calcular incluso con pocos puntos
    y = data.rolling(window=window_size, center=True, min_periods=1).mean()
    
    return y.values


In [ ]:
ventana1 = 30
ventana2 = 30
fc = 0.5

#Filtrramos tambien con el filtro paso alto


señal_filt_LP = filtro_media_movil_pandas(df['intra_LP'], ventana1)
señal_filt_PD = filtro_media_movil_pandas(df['intra_PD'], ventana2)


señal_filt_LP = detrend_mantener_nivel(señal_filt_LP)
señal_filt_PD = detrend_mantener_nivel(señal_filt_PD)

#señal_filt_LP = detrend_mantener_nivel(df['intra_LP'])
#señal_filt_PD = detrend_mantener_nivel(df['intra_PD'])


df['intra_LP_media_movil'] = señal_filt_LP
df['intra_PD_media_movil'] = señal_filt_PD


plt.figure(figsize=(12, 6))

plt.plot(df['tiempo'][550000:600000], señal_filt_PD[550000:600000], label='Intra PD Media Móvil')
plt.xlabel('Tiempo (s)')
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal - Filtro Media Móvil')



plt.figure(figsize=(12, 6))
plt.plot(df['tiempo'], señal_filt_LP, label='Intra LP Media Móvil')
plt.plot(df['tiempo'], señal_filt_PD, label='Intra PD Media Móvil')
plt.xlabel('Tiempo (s)')
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal- Después de eliminar tendencia y aplicar filtro de media móvil')

In [ ]:
from scipy.signal import find_peaks
import numpy as np
señal_LP = df['intra_LP_media_movil'].values
señal_PD = df['intra_PD_media_movil'].values


# Umbrales
umbral_inf_LP = -5.76
umbral_max_LP = -4.2



umbral_inf_PD = -8.3
umbral_max_PD = -7.1
umbral_maxU_PD = -6.6



dt = 0.1

peaks_LP, props_LP = find_peaks(señal_LP, height=umbral_max_LP, distance=150, )
hiper_indices_LP, props_hiper_LP = find_peaks(-señal_LP, height=-umbral_inf_LP, distance=3000,width=100)


peaks_PD, props_PD = find_peaks(señal_PD, height=umbral_max_PD, distance=150, )
hiper_indices_PD, props_hiper_PD = find_peaks(-señal_PD, height=-umbral_inf_PD, distance=5000,width=200)

import numpy as np

def detectar_rafagas(peaks, umbral_intra, signal, umbral_amplitud=None, min_picos=2):
    """
    Detecta el primer y último pico de cada ráfaga.
    
    Parámetros:
    -----------
    peaks            : array de índices de picos (ordenados)
    umbral_intra     : distancia máxima entre picos para considerarlos
                       de la MISMA ráfaga. Si el gap es mayor → nueva ráfaga.
    signal           : señal original (para consultar amplitudes)
    umbral_amplitud  : amplitud mínima que debe superar el pico final.
                       Si None, se usa el último pico del grupo (comportamiento original).
    min_picos        : número mínimo de picos para que sea una ráfaga válida
                       (descarta picos aislados o grupos muy pequeños)
    
    Retorna:
    --------
    rafagas: lista de dicts con 'inicio', 'fin', 'n_picos', 'picos'
    """
    
    gaps = np.diff(peaks)
    cortes = np.where(gaps > umbral_intra)[0]
    
    # Construir grupos
    grupos = []
    inicio_idx = 0
    for corte in cortes:
        grupos.append(peaks[inicio_idx : corte + 1])
        inicio_idx = corte + 1
    grupos.append(peaks[inicio_idx:])
    
    # Filtrar y construir ráfagas
    rafagas = []
    for grupo in grupos:
        if len(grupo) < min_picos:
            continue
        
        # Determinar el pico final
        if umbral_amplitud is not None:
            # Buscar el ÚLTIMO pico del grupo que supere el umbral de amplitud
            amplitudes = signal[grupo]
            indices_validos = np.where(amplitudes >= umbral_amplitud)[0]
            
            if len(indices_validos) == 0:
                # Ningún pico supera el umbral → descartamos la ráfaga
                continue
            
            pico_fin = grupo[indices_validos[-1]]   # último que supera el umbral
        else:
            pico_fin = grupo[-1]  # comportamiento original
        
        rafagas.append({
            'inicio':  grupo[0],
            'fin':     pico_fin,
            'n_picos': len(grupo),
            'picos':   grupo
        })
    
    return rafagas



# --- USO ---
# Ajusta estos dos valores según tu señal:
UMBRAL_INTRA = 800   # gap máximo entre picos de la misma ráfaga (en muestras)
MIN_PICOS    = 3     # mínimo de picos para considerar ráfaga válida

rafagas = detectar_rafagas(peaks_LP, signal = señal_LP,umbral_intra=UMBRAL_INTRA, min_picos=MIN_PICOS)

# Ver resultados
for i, r in enumerate(rafagas):
    print(f"Ráfaga {i+1:3d} | inicio: {r['inicio']:10d} | fin: {r['fin']:10d} | picos: {r['n_picos']}")

# Extraer solo los arrays de inicio y fin
fSp_indices_LP = np.array([r['inicio'] for r in rafagas])
lSp_indices_LP   = np.array([r['fin']   for r in rafagas])


# --- USO ---
# Ajusta estos dos valores según tu señal:
UMBRAL_INTRA_PD = 400   # gap máximo entre picos de la misma ráfaga (en muestras)
    # mínimo de picos para considerar ráfaga válida

rafagas = detectar_rafagas(peaks_PD, signal=señal_PD, umbral_intra=UMBRAL_INTRA_PD, min_picos=MIN_PICOS, umbral_amplitud=umbral_maxU_PD)

# Ver resultados
for i, r in enumerate(rafagas):
    print(f"Ráfaga {i+1:3d} | inicio: {r['inicio']:10d} | fin: {r['fin']:10d} | picos: {r['n_picos']}")

# Extraer solo los arrays de inicio y fin
fSp_indices_PD = np.array([r['inicio'] for r in rafagas])
lSp_indices_PD   = np.array([r['fin']   for r in rafagas])



# Dividir en tramos 

In [ ]:

num_tramos = 3
longitud_total = len(df)
tamaño_tramo = longitud_total // num_tramos

tramos_df = []
rafagas_LP_por_tramo = [] 
rafagas_PD_por_tramo = []

print("--- DIVISIÓN EN 3 TRAMOS ---")

for i in range(num_tramos):

    inicio_idx = i * tamaño_tramo
  
    fin_idx = longitud_total if i == num_tramos - 1 else (i + 1) * tamaño_tramo
    

    df_tramo = df.iloc[inicio_idx:fin_idx].copy()
    tramos_df.append(df_tramo)
    


    mascara_LP = (fSp_indices_LP >= inicio_idx) & (lSp_indices_LP < fin_idx)
    rafagas_LP_tramo = {
        'inicio': fSp_indices_LP[mascara_LP],
        'fin': lSp_indices_LP[mascara_LP]
    }
    rafagas_LP_por_tramo.append(rafagas_LP_tramo)
    

    mascara_PD = (fSp_indices_PD >= inicio_idx) & (lSp_indices_PD < fin_idx)
    rafagas_PD_tramo = {
        'inicio': fSp_indices_PD[mascara_PD],
        'fin': lSp_indices_PD[mascara_PD]
    }
    rafagas_PD_por_tramo.append(rafagas_PD_tramo)

    # Imprimir un resumen de cada tramo
    print(f"Tramo {i+1}:")
    print(f"  - Rango de índices : [{inicio_idx} : {fin_idx}]")
    print(f"  - Tiempo abarcado  : {df_tramo['tiempo'].iloc[0]} ms a {df_tramo['tiempo'].iloc[-1]} ms")
    print(f"  - Ráfagas LP total : {np.sum(mascara_LP)}")
    print(f"  - Ráfagas PD total : {np.sum(mascara_PD)}\n")

In [ ]:
def plot_signal_subplots(t_irr, x_irr, min_t, min_x, max_t, max_x, maxU_t, maxU_x,
                         window_size=5, step=5, cols=2):
    """
    Genera subplots con ventanas temporales de la señal.
    """

    min_t = np.asarray(min_t)
    min_x = np.asarray(min_x)
    max_t = np.asarray(max_t)
    max_x = np.asarray(max_x)
    maxU_t = np.asarray(maxU_t)
    maxU_x = np.asarray(maxU_x)
    

    print(f"Dimensiones:")
    print(f"  min_t: {min_t.shape}, min_x: {min_x.shape}")
    print(f"  max_t: {max_t.shape}, max_x: {max_x.shape}")
    print(f"  maxU_t: {maxU_t.shape}, maxU_x: {maxU_x.shape}")
    

    assert len(min_t) == len(min_x), f"min_t y min_x tienen diferentes tamaños: {len(min_t)} vs {len(min_x)}"
    assert len(max_t) == len(max_x), f"max_t y max_x tienen diferentes tamaños: {len(max_t)} vs {len(max_x)}"
    assert len(maxU_t) == len(maxU_x), f"maxU_t y maxU_x tienen diferentes tamaños: {len(maxU_t)} vs {len(maxU_x)}"
    
    t_total = t_irr[-1]
    n_windows = int(np.ceil(t_total / step))
    
    # Calcular filas necesarias
    rows = int(np.ceil(n_windows / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(12*cols, 4*rows))
    axes = axes.flatten() if n_windows > 1 else [axes]
    
    for i in range(n_windows):
        t_start = i * step
        t_end = min(t_start + window_size, t_total)
        
        if t_start >= t_total:
            break
        
        ax = axes[i]
  
        mask_signal = (t_irr >= t_start) & (t_irr <= t_end)
        
        indices_min = np.where((min_t >= t_start) & (min_t <= t_end))[0]
        indices_max = np.where((max_t >= t_start) & (max_t <= t_end))[0]
        indices_maxU = np.where((maxU_t >= t_start) & (maxU_t <= t_end))[0]

        ax.plot(t_irr[mask_signal], x_irr[mask_signal], alpha=0.7, label='x(t)')
        
        if len(indices_min) > 0:
            ax.plot(min_t[indices_min], min_x[indices_min], 'o', color='red', 
                   markersize=4, label='Mínimos')
        
        if len(indices_max) > 0:
            if indices_max.max() >= len(max_x):
                print(f"ERROR en ventana {i}: max índice={indices_max.max()}, len(max_x)={len(max_x)}")
                print(f"indices_max problemáticos: {indices_max[indices_max >= len(max_x)]}")
                # Filtrar índices válidos
                indices_max = indices_max[indices_max < len(max_x)]
            
            if len(indices_max) > 0:
                ax.plot(max_t[indices_max], max_x[indices_max], 'o', color='green', 
                       markersize=4, label='Máximos')
        
        if len(indices_maxU) > 0:
            if indices_maxU.max() >= len(maxU_x):
                print(f"ERROR en ventana {i}: maxU índice={indices_maxU.max()}, len(maxU_x)={len(maxU_x)}")
                indices_maxU = indices_maxU[indices_maxU < len(maxU_x)]
            
            if len(indices_maxU) > 0:
                ax.plot(maxU_t[indices_maxU], maxU_x[indices_maxU], 'o', color='orange', 
                       markersize=4, label='Máximos previos')
        
        ax.set_xlabel("Tiempo (t)")
        ax.set_ylabel("x(t)")
        ax.set_title(f"t ∈ [{t_start:.1f}, {t_end:.1f}]")
        ax.set_xlim(t_start, t_end)
        ax.grid(True)
        
        if i == 0:
            ax.legend(fontsize=8)
    
    # Ocultar subplots vacíos
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()



In [ ]:
for i in range(num_tramos):
    df_tramo = tramos_df[i]
    
    inicio_idx = i * tamaño_tramo
    fin_idx = longitud_total if i == num_tramos - 1 else (i + 1) * tamaño_tramo

    t_tramo = np.arange(len(df_tramo)) * dt 

    señal_LP_tramo = df_tramo['intra_LP_media_movil'].values

    # --- LP ---
    mascara_LP = (fSp_indices_LP >= inicio_idx) & (lSp_indices_LP < fin_idx)
    fSp_idx_tramo_LP = fSp_indices_LP[mascara_LP] - inicio_idx
    lSp_idx_tramo_LP = lSp_indices_LP[mascara_LP] - inicio_idx

    mascara_hiper_LP = (hiper_indices_LP >= inicio_idx) & (hiper_indices_LP < fin_idx)
    hiper_idx_tramo_LP = hiper_indices_LP[mascara_hiper_LP] - inicio_idx

    hiper_t   = t_tramo[hiper_idx_tramo_LP]
    hiper_amp = señal_LP_tramo[hiper_idx_tramo_LP]

    fSp_t   = t_tramo[fSp_idx_tramo_LP]
    fSp_amp = señal_LP_tramo[fSp_idx_tramo_LP]

    lSp_t   = t_tramo[lSp_idx_tramo_LP]
    lSp_amp = señal_LP_tramo[lSp_idx_tramo_LP]

    print(f"\n=== Tramo {i+1} — LP ===")
    plot_signal_subplots(
        t_tramo, señal_LP_tramo,
        hiper_t, hiper_amp,
        fSp_t,   fSp_amp,
        lSp_t,   lSp_amp,
        window_size=5000, step=5000, cols=3
    )

In [ ]:
for i in range(num_tramos):
    df_tramo = tramos_df[i]
    
    inicio_idx = i * tamaño_tramo
    fin_idx = longitud_total if i == num_tramos - 1 else (i + 1) * tamaño_tramo

    # Tiempo local del tramo (empieza en 0 para cada subplot)
    t_tramo = np.arange(len(df_tramo)) * dt  # tiempo relativo al tramo

    señal_PD_tramo = df_tramo['intra_PD_media_movil'].values

    # --- PD ---
    mascara_PD = (fSp_indices_PD >= inicio_idx) & (lSp_indices_PD < fin_idx)
    fSp_idx_tramo_PD = fSp_indices_PD[mascara_PD] - inicio_idx
    lSp_idx_tramo_PD = lSp_indices_PD[mascara_PD] - inicio_idx

    # Hiperpolarizaciones PD en este tramo
    mascara_hiper_PD = (hiper_indices_PD >= inicio_idx) & (hiper_indices_PD < fin_idx)
    hiper_idx_tramo_PD = hiper_indices_PD[mascara_hiper_PD] - inicio_idx

    # Tiempos y amplitudes en coordenadas locales del tramo
    hiper_t   = t_tramo[hiper_idx_tramo_PD]
    hiper_amp = señal_PD_tramo[hiper_idx_tramo_PD]

    fSp_t   = t_tramo[fSp_idx_tramo_PD]
    fSp_amp = señal_PD_tramo[fSp_idx_tramo_PD]

    lSp_t   = t_tramo[lSp_idx_tramo_PD]
    lSp_amp = señal_PD_tramo[lSp_idx_tramo_PD]

    print(f"\n=== Tramo {i+1} — PD ===")
    plot_signal_subplots(
        t_tramo, señal_PD_tramo,
        hiper_t, hiper_amp,
        fSp_t,   fSp_amp,
        lSp_t,   lSp_amp,
        window_size=5000, step=5000, cols=3
    )

In [ ]:


def periodo_hiper(ind_minimo, dt):
    """Calcula el periodo de las hiperpolarizaciones y guarda timestamps"""
    periodos_minimos = []
    timestamps = []
    for i in range(1, len(ind_minimo)):
        periodo = (ind_minimo[i] - ind_minimo[i-1]) * dt
        periodos_minimos.append(periodo)
        timestamps.append(ind_minimo[i-1] * dt)
    return periodos_minimos, timestamps




def hiper_to_Spikes_Intra(ind_minimo, ind_picos1, ind_picos2, dt):
    """Calcula tiempo desde mínimo hasta su primer y último pico DEL MISMO CICLO."""
    t_to_first = []
    t_to_last = []
    timestamps = []
    j = 0
    
    for i in range(len(ind_minimo)):
        t_min = ind_minimo[i]
        # Límite superior: el siguiente ciclo (para evitar medir ciclos silenciosos)
        t_min_sig = ind_minimo[i + 1] if i + 1 < len(ind_minimo) else float('inf')

        while j < len(ind_picos1) and ind_picos1[j] <= t_min:
            j += 1

        # Si hay un pico y ocurre ANTES del siguiente ciclo
        if j < len(ind_picos1) and ind_picos1[j] < t_min_sig:
            t_to_first.append((ind_picos1[j] - t_min) * dt)
            t_to_last.append((ind_picos2[j] - t_min) * dt)
            timestamps.append(t_min * dt)
            j += 1 
            
    return t_to_first, t_to_last, timestamps

def Spikes_to_hiper_Intra(ind_picos1, ind_picos2, ind_minimo, dt):
    """
    Calcula tiempo desde primer y último pico de cada ráfaga 
    hasta el SIGUIENTE mínimo ESTRICTAMENTE posterior al último pico.
    Sin regla de consumo: cada ráfaga busca independientemente.
    """
    t_first_to_hiper = []
    t_last_to_hiper  = []
    timestamps_first = []
    timestamps_last  = []

    for i in range(len(ind_picos1)):
        t_first = ind_picos1[i]
        t_last  = ind_picos2[i]

        candidatos = ind_minimo[ind_minimo > t_last]

        if len(candidatos) == 0:
            continue  #

        t_hiper = candidatos[0]  

        t_first_to_hiper.append((t_hiper - t_first) * dt)
        t_last_to_hiper.append((t_hiper - t_last)   * dt)
        timestamps_first.append(t_first * dt)
        timestamps_last.append(t_last   * dt)

    return t_first_to_hiper, t_last_to_hiper, timestamps_first, timestamps_last




def FirstSpike_to_hiper_Inter(ind_picos1, ind_minimo_n2, dt):
    """
    Primer pico N1 → siguiente hiperpolarización N2 
    estrictamente posterior al primer pico.
    Sin regla de consumo.
    """
    t_to_hiper = []
    timestamps  = []

    for t_first in ind_picos1:
        candidatos = ind_minimo_n2[ind_minimo_n2 > t_first]

        if len(candidatos) == 0:
            continue

        t_to_hiper.append((candidatos[0] - t_first) * dt)
        timestamps.append(t_first * dt)

    return t_to_hiper, timestamps


def LastSpike_to_hiper_Inter(ind_picos2, ind_minimo_n2, dt):
    """
    Último pico N1 → siguiente hiperpolarización N2 
    estrictamente posterior al último pico.
    Sin regla de consumo.
    """
    t_to_hiper = []
    timestamps  = []

    for t_last in ind_picos2:
        candidatos = ind_minimo_n2[ind_minimo_n2 > t_last]

        if len(candidatos) == 0:
            continue

        t_to_hiper.append((candidatos[0] - t_last) * dt)
        timestamps.append(t_last * dt)

    return t_to_hiper, timestamps

def hiperN1_to_N2(ind_minimo_n1, ind_minimo_n2, dt, tolerancia_max_ms=1800.0, tolerancia_min_ms=50.0):
    """
    Desfase de fase: desde mínimo N1 hasta el siguiente mínimo N2.
    Búsqueda vectorizada sin regla de consumo.
    tolerancia_max_ms: descarta saltos que superen un periodo máximo.
    tolerancia_min_ms: descarta emparejamientos físicamente imposibles (< tolerancia).
    """
    tiempos    = []
    timestamps = []

    ind_minimo_n2  = np.asarray(ind_minimo_n2)
    tolerancia_max_idx = int(tolerancia_max_ms / dt)
    tolerancia_min_idx = int(tolerancia_min_ms / dt)

    for t_n1 in ind_minimo_n1:

        candidatos = ind_minimo_n2[ind_minimo_n2 > (t_n1 + tolerancia_min_idx)]

        if len(candidatos) == 0:
            continue

        t_n2 = candidatos[0]

        if (t_n2 - t_n1) > tolerancia_max_idx:
            continue

        tiempos.append((t_n2 - t_n1) * dt)
        timestamps.append(t_n1 * dt)

    return tiempos, timestamps

def hiperN1_to_SpikesN2(ind_minimo_n1, ind_picos1_n2, ind_picos2_n2, dt, 
                         tolerancia_max_ms=1800.0, tolerancia_min_ms=50.0):
    """
    Tiempo desde mínimo N1 hasta la siguiente ráfaga N2 (primer y último pico).
    - Búsqueda vectorizada sin regla de consumo.
    - tolerancia_max_ms: descarta saltos que superen un periodo máximo.
    - tolerancia_min_ms: descarta emparejamientos físicamente imposibles.
    """
    t_hiper_to_first = []
    t_hiper_to_last  = []
    timestamps       = []

    ind_picos1_n2      = np.asarray(ind_picos1_n2)
    ind_picos2_n2      = np.asarray(ind_picos2_n2)
    tolerancia_max_idx = int(tolerancia_max_ms / dt)
    tolerancia_min_idx = int(tolerancia_min_ms / dt)

    for t_min in ind_minimo_n1:

        # Candidatos: primer pico posterior a t_min + tolerancia mínima
        mask = ind_picos1_n2 > (t_min + tolerancia_min_idx)
        if not np.any(mask):
            continue

        idx_rafaga = np.where(mask)[0][0]
        t_first    = ind_picos1_n2[idx_rafaga]
        t_last     = ind_picos2_n2[idx_rafaga]

        if (t_first - t_min) > tolerancia_max_idx:
            continue

        t_hiper_to_first.append((t_first - t_min) * dt)
        t_hiper_to_last.append( (t_last  - t_min) * dt)
        timestamps.append(t_min * dt)

    return t_hiper_to_first, t_hiper_to_last, timestamps

def SpikesN1_to_hiperN2(ind_picos1, ind_picos2, ind_minimo_n2, dt, tolerancia_min_ms=50.0):
    """
    Versión cruzada: hiper N2 estrictamente posterior al último pico N1.
    tolerancia_min_ms: descarta emparejamientos donde el intervalo es 
                       físicamente imposible (< tolerancia).
    """
    t_first_to_hiper = []
    t_last_to_hiper  = []
    timestamps_first = []
    timestamps_last  = []

    ind_minimo_n2  = np.asarray(ind_minimo_n2)
    tolerancia_idx = int(tolerancia_min_ms / dt)  # convertir ms a muestras

    for i in range(len(ind_picos1)):
        t_first = ind_picos1[i]
        t_last  = ind_picos2[i]

        # Mínimo debe ser posterior al último pico + tolerancia mínima
        candidatos = ind_minimo_n2[ind_minimo_n2 > (t_last + tolerancia_idx)]

        if len(candidatos) == 0:
            continue

        t_hiper = candidatos[0]

        t_first_to_hiper.append((t_hiper - t_first) * dt)
        t_last_to_hiper.append( (t_hiper - t_last)  * dt)
        timestamps_first.append(t_first * dt)
        timestamps_last.append( t_last  * dt)

    return t_first_to_hiper, t_last_to_hiper, timestamps_first, timestamps_last

In [ ]:
# Importar los intervalos que es otro archivo

ruta_intervalos = r"C:\Users\huozh\Desktop\Proyectos Universidad\Universidad\TFG\Datos_Robot_Articulo\Submission\FLC-Hybrot_Data\Fig1\data_Control_17h55m39s-12-May.txt"

df_bursts = pd.read_csv(
    ruta_intervalos,      
    sep=r'\t',  # Separador por tabulaciones
    header=0   # Primera fila son los encabezados
)

print(df_bursts.head())
print(f"\nDimensiones: {df_bursts.shape}")
print(f"\nColumnas: {df_bursts.columns.tolist()}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# 1. Periodos Hlp y Hpd
periodos_hiper_n1, t_periodos_hiper_n1 = periodo_hiper(hiper_indices_LP, dt)
periodos_hiper_n2, t_periodos_hiper_n2 = periodo_hiper(hiper_indices_PD, dt)

# 2. Intervalos Intraneuronales Puros (Hiper -> Spikes)
hiper_to_firstspike_n1, hiper_to_lastspike_n1, t_hiper_n1 = hiper_to_Spikes_Intra(
    hiper_indices_LP, fSp_indices_LP, lSp_indices_LP, dt
)
hiper_to_firstspike_n2, hiper_to_lastspike_n2, t_hiper_n2 = hiper_to_Spikes_Intra(
    hiper_indices_PD, fSp_indices_PD, lSp_indices_PD, dt
)
# Los timestamps son los mismos para first y last (mismo ciclo)
t_hiper_to_lastspike_n1 = t_hiper_n1
t_hiper_to_lastspike_n2 = t_hiper_n2

# 3. Intervalos Intraneuronales Compuestos (Spikes -> Hiper misma neurona)
# ── NUEVO: ahora devuelve 4 valores y usa búsqueda vectorizada sin regla de consumo
firstspike_hiper_n1, lastspike_hiper_n1, t_firstspike_hiper_n1, t_lastspike_hiper_n1 = Spikes_to_hiper_Intra(
    fSp_indices_LP, lSp_indices_LP, hiper_indices_LP, dt
)
firstspike_hiper_n2, lastspike_hiper_n2, t_firstspike_hiper_n2, t_lastspike_hiper_n2 = Spikes_to_hiper_Intra(
    fSp_indices_PD, lSp_indices_PD, hiper_indices_PD, dt
)

# 4. Desfases Base (Hiper -> Hiper cruzado)
hiperN1_to_N2_times, t_hiperN1_to_N2_times = hiperN1_to_N2(hiper_indices_LP, hiper_indices_PD, dt)
hiperN2_to_N1_times, t_hiperN2_to_N1_times = hiperN1_to_N2(hiper_indices_PD, hiper_indices_LP, dt)

# 5. Intervalos Compuestos Cruzados (Spikes -> Hiper cruzado)
# ── NUEVO: SpikesN1_to_hiperN2 devuelve 4 valores con timestamps separados
firstspikeN1_to_hiperN2_times, lastspikeN1_to_hiperN2_times, \
t_firstspikeN1_to_hiperN2_times, t_lastspikeN1_to_hiperN2_times = SpikesN1_to_hiperN2(
    fSp_indices_LP, lSp_indices_LP, hiper_indices_PD, dt
)
firstspikeN2_to_hiperN1_times, lastspikeN2_to_hiperN1_times, \
t_firstspikeN2_to_hiperN1_times, t_lastspikeN2_to_hiperN1_times = SpikesN1_to_hiperN2(
    fSp_indices_PD, lSp_indices_PD, hiper_indices_LP, dt
)

# 6. Intervalos Compuestos Cruzados (Hiper -> Spikes cruzado)
# ── hiperN1_to_SpikesN2 devuelve 3 valores; timestamp es compartido para first y last
hiperN1_to_FirstSpikeN2_times, hiperN1_to_LastSpikeN2_times, \
t_hiperN1_to_SpikesN2_times = hiperN1_to_SpikesN2(
    hiper_indices_LP, fSp_indices_PD, lSp_indices_PD, dt
)
t_hiperN1_to_FirstSpikeN2_times = t_hiperN1_to_SpikesN2_times
t_hiperN1_to_LastSpikeN2_times  = t_hiperN1_to_SpikesN2_times

hiperN2_to_FirstSpikeN1_times, hiperN2_to_LastSpikeN1_times, \
t_hiperN2_to_SpikesN1_times = hiperN1_to_SpikesN2(
    hiper_indices_PD, fSp_indices_LP, lSp_indices_LP, dt
)
t_hiperN2_to_FirstSpikeN1_times = t_hiperN2_to_SpikesN1_times
t_hiperN2_to_LastSpikeN1_times  = t_hiperN2_to_SpikesN1_times




# Referencia base: tiempo de las hiperpolarizaciones LP (usamos t_periodos_hiper_n1)
df_global = pd.DataFrame({'Time': t_periodos_hiper_n1}).sort_values('Time').reset_index(drop=True)

# Lista de métricas calculadas arriba para el merge, mapeadas correctamente
datos_a_unir = [
    ("Periodo Hlp(ms)",        periodos_hiper_n1,              t_periodos_hiper_n1),
    ("Periodo Hpd(ms)",        periodos_hiper_n2,              t_periodos_hiper_n2),
    # Intraneuronales Hiper→Spikes  (timestamp = t_hiper_n1/n2)
    ('Intervalo HlpFSlp(ms)',  hiper_to_firstspike_n1,         t_hiper_n1),
    ('Intervalo HlpLSlp(ms)',  hiper_to_lastspike_n1,          t_hiper_n1),
        ('Intervalo HpdFSpd(ms)',  hiper_to_firstspike_n2,         t_hiper_n2),
        ('Intervalo HpdLSpd(ms)',  hiper_to_lastspike_n2,          t_hiper_n2),
        # Intraneuronales Spikes→Hiper  (timestamps separados)
        ('Intervalo FSlpHlp(ms)',  firstspike_hiper_n1,            t_firstspike_hiper_n1),
        ('Intervalo LSlpHlp(ms)',  lastspike_hiper_n1,             t_lastspike_hiper_n1),
        ('Intervalo FSpdHpd(ms)',  firstspike_hiper_n2,            t_firstspike_hiper_n2),
        ('Intervalo LSpdHpd(ms)',  lastspike_hiper_n2,             t_lastspike_hiper_n2),
        # Hiper→Hiper cruzado
        ('Intervalo HlpHpd(ms)',   hiperN1_to_N2_times,            t_hiperN1_to_N2_times),
        ('Intervalo HpdHlp(ms)',   hiperN2_to_N1_times,            t_hiperN2_to_N1_times),
        # Spikes→Hiper cruzado (timestamps separados por first/last)
        ('Intervalo FSlpHpd(ms)',  firstspikeN1_to_hiperN2_times,  t_firstspikeN1_to_hiperN2_times),
        ('Intervalo LSlpHpd(ms)',  lastspikeN1_to_hiperN2_times,   t_lastspikeN1_to_hiperN2_times),
        ('Intervalo FSpdHlp(ms)',  firstspikeN2_to_hiperN1_times,  t_firstspikeN2_to_hiperN1_times),
        ('Intervalo LSpdHlp(ms)',  lastspikeN2_to_hiperN1_times,   t_lastspikeN2_to_hiperN1_times),
        # Hiper→Spikes cruzado (timestamp compartido)
        ('Intervalo HlpFSpd(ms)',  hiperN1_to_FirstSpikeN2_times,  t_hiperN1_to_FirstSpikeN2_times),
        ('Intervalo HlpLSpd(ms)',  hiperN1_to_LastSpikeN2_times,   t_hiperN1_to_LastSpikeN2_times),
        ('Intervalo HpdFSlp(ms)',  hiperN2_to_FirstSpikeN1_times,  t_hiperN2_to_FirstSpikeN1_times),
        ('Intervalo HpdLSlp(ms)',  hiperN2_to_LastSpikeN1_times,   t_hiperN2_to_LastSpikeN1_times),
    ]

# Unimos los datos calculados de hiperpolarizaciones
for nombre_col, valores, tiempos in datos_a_unir:
    if len(valores) > 0:
        df_temp = pd.DataFrame({'Time': tiempos, nombre_col: valores}).sort_values('Time')
        df_global = pd.merge_asof(df_global, df_temp, on='Time', direction='nearest', tolerance=5000)

# ----------------------------------------------------------
# INTEGRACIÓN DE DATOS EXTERNOS (df_bursts)
# ----------------------------------------------------------
# Extraemos las columnas nuevas de tu tabla 'df_bursts'
df_extra = df_bursts[['t1LP', 'fst2fstLP', 'LPPD1spkperiod']].copy()
df_extra = df_extra.sort_values('t1LP')

# Las unimos a la tabla maestra
df_global = pd.merge_asof(
    df_global, 
    df_extra, 
    left_on='Time', 
    right_on='t1LP', 
    direction='nearest', 
    tolerance=5000
)

# Limpieza y estabilidad
# El dropna() asegura que solo grafiquemos ciclos donde TODAS las variables coinciden
df_global_limpio = df_global.dropna().reset_index(drop=True)



# ============================================================
# 3. SEGMENTACIÓN Y PLOTEO POR TRAMOS
# ============================================================
num_tramos = 4
t_min, t_max = df_global_limpio['Time'].min(), df_global_limpio['Time'].max()
paso = (t_max - t_min) / num_tramos

for i in range(num_tramos):
    inicio = t_min + i * paso
    fin = inicio + paso
    
    # Recorte del tramo
    df_tramo = df_global_limpio[(df_global_limpio['Time'] >= inicio) & (df_global_limpio['Time'] < fin)].copy()
    
    if len(df_tramo) < 10: 
        print(f"Tramo {i+1} con pocos datos, se omite.")
        continue

    print(f"Generando Pairplot Tramo {i+1} (n={len(df_tramo)} ciclos)...")
    
    # Categoría temporal interna
    df_tramo['Time_Category'] = pd.qcut(df_tramo['Time'], 3, labels=['Inicio', 'Medio', 'Final'])
    
    # Definimos qué variables queremos comparar (ajusta esta lista a tu gusto)
    
    g = sns.pairplot(
        df_tramo,
        hue='Time_Category',
        palette=['blue', 'orange', 'red'],
        plot_kws={'alpha': 0.4, 's': 20},
        diag_kind='kde'
    )
    
    g.fig.suptitle(f'Segmento {i+1} | {inicio:.0f} - {fin:.0f} ms', y=1.02, fontsize=14)
    plt.show()

## exportar .txt

In [ ]:
# ============================================================
# EXPORTACIÓN A .TXT SIN MAPEO DE NOMBRES
# ============================================================

# 1. Definir el número de tramos y los límites temporales
num_tramos = 4
t_min = df_global_limpio['Time'].min()
t_max = df_global_limpio['Time'].max()
paso = (t_max - t_min) / num_tramos

for i in range(num_tramos):
    t_inicio = t_min + i * paso
    t_fin = t_inicio + paso
    
    # 2. Extraer el tramo de la tabla maestra ya limpia
    df_export = df_global_limpio[(df_global_limpio['Time'] >= t_inicio) & (df_global_limpio['Time'] < t_fin)].copy()
    
    if df_export.empty:
        print(f"Tramo {i+1} vacío, saltando...")
        continue
        
    # 3. Insertar la columna #burst al principio (índice 0, 1, 2...)
    # Esto es útil para que herramientas externas identifiquen cada ciclo
    df_export.insert(0, '#burst', range(len(df_export)))
    
    # 4. Guardar como archivo .txt separado por tabuladores
    nombre_archivo = f'intervalos_originales_tramo_{i+1}.txt'
    
    # Usamos index=False para no añadir la columna de índices de pandas
    # Usamos sep='\t' para mantener el formato de columnas alineadas
    df_export.to_csv(nombre_archivo, sep='\t', index=False)
    
    print(f"Archivo exportado: {nombre_archivo} | Ciclos: {len(df_export)}")